In [1]:
import pandas as pd
import numpy as np
import joblib
import json
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

print("Phase 7: Automated Retraining Pipeline")
print(f"Run timestamp: {datetime.now().isoformat()}")

Phase 7: Automated Retraining Pipeline
Run timestamp: 2026-05-11T13:19:23.547360


In [2]:
# Load latest data (in production, this comes from your data warehouse)
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges'])
df = df.drop('customerID', axis=1)

# Load previous model for comparison
old_pipeline = joblib.load('../models/phase4_logistic_regression.pkl')

print(f"Fresh data: {df.shape}")
print(f"Old model loaded")

Fresh data: (7032, 20)
Old model loaded


In [3]:
X = df.drop('Churn', axis=1)
y = df['Churn'].map({'No': 0, 'Yes': 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
categorical_features = [c for c in X.columns if c not in numeric_features]

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
])

new_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
])

new_pipeline.fit(X_train, y_train)

# Score new model
y_pred_new = new_pipeline.predict(X_test)
y_proba_new = new_pipeline.predict_proba(X_test)[:, 1]

new_metrics = {
    'accuracy': accuracy_score(y_test, y_pred_new),
    'recall': recall_score(y_test, y_pred_new),
    'roc_auc': roc_auc_score(y_test, y_proba_new),
    'timestamp': datetime.now().isoformat()
}

print("New model trained")
print(f"Accuracy: {new_metrics['accuracy']:.4f}")
print(f"Recall:   {new_metrics['recall']:.4f}")
print(f"ROC-AUC:  {new_metrics['roc_auc']:.4f}")

New model trained
Accuracy: 0.7264
Recall:   0.7941
ROC-AUC:  0.8353


In [4]:
# Score old model on same test set for fair comparison
y_pred_old = old_pipeline.predict(X_test)
y_proba_old = old_pipeline.predict_proba(X_test)[:, 1]

old_metrics = {
    'accuracy': accuracy_score(y_test, y_pred_old),
    'recall': recall_score(y_test, y_pred_old),
    'roc_auc': roc_auc_score(y_test, y_proba_old)
}

print("=" * 50)
print("CHAMPION (Old) vs CHALLENGER (New)")
print("=" * 50)
print(f"{'Metric':<12} {'Old':>8} {'New':>8} {'Delta':>8}")
print("-" * 40)
for metric in ['accuracy', 'recall', 'roc_auc']:
    delta = new_metrics[metric] - old_metrics[metric]
    winner = "✅ NEW" if delta > 0 else "⚠️ OLD"
    print(f"{metric:<12} {old_metrics[metric]:>8.4f} {new_metrics[metric]:>8.4f} {delta:>+8.4f} {winner}")

# Promote if new model wins on recall (your business metric)
PROMOTE_THRESHOLD = 0.01  # New must be at least 1% better
should_promote = new_metrics['recall'] > old_metrics['recall'] + PROMOTE_THRESHOLD

print(f"\n{'🚀 PROMOTE NEW MODEL' if should_promote else '⏸️ KEEP OLD MODEL'}")

CHAMPION (Old) vs CHALLENGER (New)
Metric            Old      New    Delta
----------------------------------------
accuracy       0.7264   0.7264  +0.0000 ⚠️ OLD
recall         0.7941   0.7941  +0.0000 ⚠️ OLD
roc_auc        0.8353   0.8353  +0.0000 ⚠️ OLD

⏸️ KEEP OLD MODEL


In [5]:
import os
import shutil

if should_promote:
    # Archive old model with timestamp
    archive_name = f"../models/archive/phase4_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pkl"
    os.makedirs('../models/archive', exist_ok=True)
    shutil.copy('../models/phase4_logistic_regression.pkl', archive_name)
    
    # Promote new model
    joblib.dump(new_pipeline, '../models/phase4_logistic_regression.pkl')
    
    # Save metrics log
    with open('../models/metrics_log.jsonl', 'a') as f:
        f.write(json.dumps(new_metrics) + '\n')
    
    print(f"Old model archived: {archive_name}")
    print("New model promoted to production")
else:
    print("New model discarded. Old model remains in production.")

New model discarded. Old model remains in production.


In [6]:
import os
import shutil

if should_promote:
    # Archive old model with timestamp
    archive_name = f"../models/archive/phase4_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pkl"
    os.makedirs('../models/archive', exist_ok=True)
    shutil.copy('../models/phase4_logistic_regression.pkl', archive_name)
    
    # Promote new model
    joblib.dump(new_pipeline, '../models/phase4_logistic_regression.pkl')
    
    # Save metrics log
    with open('../models/metrics_log.jsonl', 'a') as f:
        f.write(json.dumps(new_metrics) + '\n')
    
    print(f"Old model archived: {archive_name}")
    print("New model promoted to production")
else:
    print("New model discarded. Old model remains in production.")

New model discarded. Old model remains in production.
